# Sentiment Analysis
## Working on Internet Movie Database (IMBd), building a predictor that can distinguish between positive and negative reviews

Obtaining the dataset
Were not abel to accsess http://ai.stanford.edu/~amaas/data/sentiment/ for some reason, so obtained the dataset from: https://huggingface.co/datasets/stanfordnlp/imdb instead

In [1]:
from datasets import load_dataset
ds = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:
import numpy as np
import pandas as pd

### Decompressing alle reviews:
In to one single dataframe object with 50 000 reveiews, with label/class 0 for negative and 1 for positive

In [8]:
# compressing all
df = pd.concat([ds["train"].to_pandas(), ds["test"].to_pandas()], ignore_index=True)
df = df.rename(columns={"text": "review", "label": "sentiment"})

# randomly shuffle
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))

# to csv file
df.to_csv("data/movie_data.csv", index=False, encoding="utf-8")


In [9]:
# checking if everything is correct
df = pd.read_csv("data/movie_data.csv", encoding="utf-8")
print(df.shape) 
df.head(3)

(50000, 2)


,review,sentiment
0,If your idea of entertainment is watching grap...,0
1,An intriguingly bold film weaves the seemingly...,1
2,"With ""Batman Returns"", Tim Burton succumbed to...",1


### Bag of words model
Categorical data such as text needs to be transformed into numerical form.
Bag-of-words lets us represent text as numerical vectors.
Idea behind bag-of-words:
1. Create a vocubalary of unique tokens (for example words) from the entire set of documents
2. Construct a feature vector from each document that contains the counts of how often each word occurs in the particular document

In [ ]:
# use CountVectorizer to consturct a bag-of-words (can be used on array of text, entire documents, sentnces)
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()

docs = np.array(["The sun is shining",
                 "The weather is sweet",
                 "The sun is shining, the weather is sweet"
                 "and one and one is two"])

# caling fit_transform, created vocubalary of the bag-of-words and transromed into three sentences into sparse vectors
bag = count.fit_transform(docs)

# prints contents, of the vocubulary
# stored in a dict
print(count.vocabulary_)

{'the': 7, 'sun': 4, 'is': 1, 'shining': 3, 'weather': 9, 'sweet': 5, 'sweetand': 6, 'one': 2, 'and': 0, 'two': 8}


In [13]:
# print the feature vectors of the sentences
print(bag.toarray())

[[0 1 0 1 1 0 0 1 0 0]
 [0 1 0 0 0 1 0 1 0 1]
 [1 3 2 1 1 0 1 2 1 1]]


This feature vectors are called the raw term frequencies tf(t,d), the number of time a term t, occurs in a document, d.
The bag-of-word model here is called 1-gram, because each item or token in the vocubulary represents a single word. In generell it's called n-grams model, the choice of n in n-gram model depends on the application. \
**Example**
- 1-gram: "the", "sun", "is", "shinning"
- 2-gram: "the sun", "sun is", "is shinning"